# Riot API Match Collection

This notebook collects Ranked Solo/Duo match and timeline data from the Riot Games API for later feature engineering and win-condition analysis.

# 1. API Setup

In [32]:
import json
import os
import random
import time

import requests
from dotenv import load_dotenv

load_dotenv(override=True)

RIOT_API_KEY = os.getenv("RIOT_API_KEY")

headers = {
    "X-Riot-Token": RIOT_API_KEY
}

print("API key loaded:", RIOT_API_KEY is not None)
print("Key length:", len(RIOT_API_KEY) if RIOT_API_KEY else 0)

API key loaded: True
Key length: 42


# 2. Configuration

In [33]:
ROUTING = "americas"
PLATFORM = "na1"

TARGET_PATCH = "16.18"
TARGET_QUEUE = 420
RANKED_QUEUE = "RANKED_SOLO_5x5"

TARGET_N = 100
NUM_MASTER_PLAYERS = 20
MATCHES_PER_PLAYER = 50

# 3. Riot API Helpers

The collection flow is Master League-V4 leaderboard → PUUID → Match-V5.

In [34]:
def riot_get(url, params=None):
    while True:
        response = requests.get(
            url,
            headers=headers,
            params=params
        )

        if response.status_code == 429:
            wait_time = int(
                response.headers.get("Retry-After", 10)
            )

            print(
                f"Rate limited. Waiting {wait_time} seconds..."
            )

            time.sleep(wait_time + 1)
            continue

        response.raise_for_status()
        return response.json()

In [35]:
def get_match_ids(puuid, count=20):
    url = (
        f"https://{ROUTING}.api.riotgames.com/"
        f"lol/match/v5/matches/by-puuid/{puuid}/ids"
    )

    params = {
        "queue": TARGET_QUEUE,
        "start": 0,
        "count": count
    }

    return riot_get(url, params=params)


def get_match(match_id):
    url = (
        f"https://{ROUTING}.api.riotgames.com/"
        f"lol/match/v5/matches/{match_id}"
    )

    return riot_get(url)


def get_timeline(match_id):
    url = (
        f"https://{ROUTING}.api.riotgames.com/"
        f"lol/match/v5/matches/{match_id}/timeline"
    )

    return riot_get(url)


def get_patch(match_data):
    version = match_data["info"]["gameVersion"]
    parts = version.split(".")
    return f"{parts[0]}.{parts[1]}"


def get_random_master_puuids(n=NUM_MASTER_PLAYERS):
    url = (
        f"https://{PLATFORM}.api.riotgames.com/"
        f"lol/league/v4/masterleagues/by-queue/{RANKED_QUEUE}"
    )

    data = riot_get(url)
    entries = data["entries"]

    random.shuffle(entries)

    puuids = []

    for entry in entries:
        puuid = entry.get("puuid")

        if puuid:
            puuids.append(puuid)

        if len(puuids) >= n:
            break

    return puuids

# 4. Sample Master-Tier Players

The source pool is randomly sampled from the NA Master-tier Ranked Solo/Duo league. No personal account data is used.

In [36]:
master_puuids = get_random_master_puuids()

if not master_puuids:
    raise RuntimeError("No Master-tier players were sampled. Check the API key and platform routing.")

print(f"Master players sampled: {len(master_puuids)}")

Master players sampled: 20


# 5. Gather Candidate Match IDs

Match-V5 is first restricted to Ranked Solo/Duo; duplicate match IDs across sampled players are removed.

In [37]:
candidate_matches = set()

for index, puuid in enumerate(master_puuids, start=1):
    try:
        match_ids = get_match_ids(puuid,count=MATCHES_PER_PLAYER)
        candidate_matches.update(match_ids)
        print(f"Player {index}/{len(master_puuids)}: {len(match_ids)} match IDs")
    except requests.HTTPError as error:
        print(f"Skipping player {index}: {error}")

    time.sleep(0.1)

candidate_matches = list(candidate_matches)
random.shuffle(candidate_matches)
print(f"Unique candidate matches: {len(candidate_matches)}")

Player 1/20: 50 match IDs
Player 2/20: 50 match IDs
Player 3/20: 50 match IDs
Player 4/20: 50 match IDs
Player 5/20: 50 match IDs
Player 6/20: 50 match IDs
Player 7/20: 50 match IDs
Player 8/20: 50 match IDs
Player 9/20: 50 match IDs
Player 10/20: 50 match IDs
Player 11/20: 50 match IDs
Player 12/20: 50 match IDs
Player 13/20: 50 match IDs
Player 14/20: 50 match IDs
Player 15/20: 50 match IDs
Player 16/20: 50 match IDs
Player 17/20: 50 match IDs
Player 18/20: 50 match IDs
Player 19/20: 50 match IDs
Player 20/20: 50 match IDs
Unique candidate matches: 995


In [38]:
selected_matches = []
selected_match_data = {}

for match_id in candidate_matches:
    if len(selected_matches) >= TARGET_N:
        break

    try:
        match_data = get_match(match_id)
    except requests.HTTPError as error:
        print("Skipping:", match_id, error)
        continue

    if get_patch(match_data) == TARGET_PATCH:
        selected_matches.append(match_id)
        selected_match_data[match_id] = match_data

        print(
            f"{len(selected_matches)}/{TARGET_N}",
            match_id,
            get_patch(match_data)
        )

    time.sleep(0.1)

print("\nSelected matches:", len(selected_matches))

1/100 NA1_5645166601 16.18
2/100 NA1_5643834815 16.18
3/100 NA1_5645278764 16.18
4/100 NA1_5641963927 16.18
5/100 NA1_5641280226 16.18
6/100 NA1_5640304474 16.18
7/100 NA1_5640914076 16.18
8/100 NA1_5639425071 16.18
9/100 NA1_5645095750 16.18
10/100 NA1_5643958297 16.18
11/100 NA1_5646599249 16.18
12/100 NA1_5641048770 16.18
13/100 NA1_5643940265 16.18
14/100 NA1_5640002630 16.18
15/100 NA1_5645347612 16.18
16/100 NA1_5646894770 16.18
17/100 NA1_5645653892 16.18
18/100 NA1_5639870977 16.18
19/100 NA1_5643645500 16.18
20/100 NA1_5642605414 16.18
21/100 NA1_5646584801 16.18
22/100 NA1_5641817389 16.18
23/100 NA1_5645436028 16.18
24/100 NA1_5640152258 16.18
25/100 NA1_5641455584 16.18
26/100 NA1_5643554578 16.18
27/100 NA1_5639823719 16.18
28/100 NA1_5644002556 16.18
Rate limited. Waiting 90 seconds...
29/100 NA1_5644709968 16.18
30/100 NA1_5640886232 16.18
31/100 NA1_5642833406 16.18
32/100 NA1_5645005461 16.18
33/100 NA1_5645124226 16.18
34/100 NA1_5639595460 16.18
35/100 NA1_5646523211

# 6. Filter, Download, and Save Data

Each candidate is checked against the target patch and queue before its match and timeline payloads are saved as a pair under `data/raw/`.

In [39]:
MATCH_OUTPUT_DIR = "data/raw/matches"
TIMELINE_OUTPUT_DIR = "data/raw/timelines"

os.makedirs(MATCH_OUTPUT_DIR, exist_ok=True)
os.makedirs(TIMELINE_OUTPUT_DIR, exist_ok=True)

for match_id in selected_matches:
    match_data = selected_match_data[match_id]
    match_path = os.path.join(MATCH_OUTPUT_DIR, f"{match_id}.json")
    timeline_path = os.path.join(TIMELINE_OUTPUT_DIR, f"{match_id}.json")

    if os.path.exists(match_path):
        print(f"Match already exists, skipping: {match_id}")
    else:
        with open(match_path, "w") as file:
            json.dump(match_data, file)

    if os.path.exists(timeline_path):
        print(f"Timeline already exists, skipping: {match_id}")
        continue

    try:
        timeline_data = get_timeline(match_id)
    except requests.HTTPError as error:
        print(f"Skipping timeline for {match_id}: {error}")
        continue

    with open(timeline_path, "w") as file:
        json.dump(timeline_data, file)

    print(f"Saved match and timeline: {match_id}")
    time.sleep(0.1)

print(f"Completed collection: {len(selected_matches)} selected matches processed.")

Saved match and timeline: NA1_5645166601
Saved match and timeline: NA1_5643834815
Saved match and timeline: NA1_5645278764
Saved match and timeline: NA1_5641963927
Saved match and timeline: NA1_5641280226
Saved match and timeline: NA1_5640304474
Saved match and timeline: NA1_5640914076
Saved match and timeline: NA1_5639425071
Saved match and timeline: NA1_5645095750
Saved match and timeline: NA1_5643958297
Saved match and timeline: NA1_5646599249
Saved match and timeline: NA1_5641048770
Saved match and timeline: NA1_5643940265
Saved match and timeline: NA1_5640002630
Saved match and timeline: NA1_5645347612
Saved match and timeline: NA1_5646894770
Saved match and timeline: NA1_5645653892
Saved match and timeline: NA1_5639870977
Saved match and timeline: NA1_5643645500
Saved match and timeline: NA1_5642605414
Saved match and timeline: NA1_5646584801
Saved match and timeline: NA1_5641817389
Saved match and timeline: NA1_5645436028
Saved match and timeline: NA1_5640152258
Saved match and 

# 7. Inspect a Saved Match

Use this optional check to confirm the saved match metadata, participant count, and timeline structure.

In [40]:
if not selected_matches:
    raise RuntimeError("No matches were saved, so there is nothing to inspect.")

example_match_id = selected_matches[0]

with open(f"{MATCH_OUTPUT_DIR}/{example_match_id}.json") as file:
    example_match = json.load(file)

with open(f"{TIMELINE_OUTPUT_DIR}/{example_match_id}.json") as file:
    example_timeline = json.load(file)

match_info = example_match["info"]
print(f"Match ID: {example_match['metadata']['matchId']}")
print(f"Queue ID: {match_info['queueId']}")
print(f"Patch: {get_patch(example_match)}")
print(f"Duration: {match_info['gameDuration'] / 60:.1f} minutes")
print(f"Participants: {len(match_info['participants'])}")
print(f"Timeline frames: {len(example_timeline['info']['frames'])}")

Match ID: NA1_5645166601
Queue ID: 420
Patch: 16.18
Duration: 31.8 minutes
Participants: 10
Timeline frames: 33


In [41]:
for player in match_info["participants"]:
    print(
        f"{player['teamPosition'] or 'UNKNOWN':<7} "
        f"{player['championName']:<16} "
        f"K/D/A: {player['kills']}/{player['deaths']}/{player['assists']} "
        f"Win: {player['win']}"
    )

print("First five frame timestamps:", [frame["timestamp"] for frame in example_timeline["info"]["frames"][:5]])

TOP     Ornn             K/D/A: 4/8/7 Win: False
JUNGLE  Nunu             K/D/A: 2/7/11 Win: False
MIDDLE  Fiora            K/D/A: 0/14/2 Win: False
BOTTOM  Kaisa            K/D/A: 17/5/0 Win: False
UTILITY Yuumi            K/D/A: 1/6/18 Win: False
TOP     Udyr             K/D/A: 9/4/5 Win: True
JUNGLE  Zed              K/D/A: 15/4/8 Win: True
MIDDLE  Ziggs            K/D/A: 9/5/11 Win: True
BOTTOM  Zeri             K/D/A: 6/3/13 Win: True
UTILITY Blitzcrank       K/D/A: 1/8/18 Win: True
First five frame timestamps: [0, 60028, 120038, 180048, 240075]


# 8. Collection Summary

In [42]:
print("Patch:", TARGET_PATCH)
print("Queue:", TARGET_QUEUE)
print("Master players sampled:", len(master_puuids))
print("Candidate matches:", len(candidate_matches))
print("Selected matches:", len(selected_matches))

Patch: 16.18
Queue: 420
Master players sampled: 20
Candidate matches: 995
Selected matches: 100
